In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.drop_duplicates(inplace=True)

In [5]:
df.shape

(49582, 2)

## Pre-Processing

### 1.Converting to lowercase

In [6]:
df["review"] = df["review"].str.lower() 

### 2. Removing the URLs

In [7]:
import re

In [8]:
def remove_urls(text):
    text = re.sub(r"http\S+", "",text) #(pattern, replace,string)
    return text
    
df["review"] = df["review"].apply(remove_urls)   

### 3. Removing Punctuations

In [9]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "",text) # A-Z,a-z,0-9
    return text

df["review"] = df["review"].apply(remove_punctuations)     

### 4. Removing HTML

In [10]:
def remove_html(text):
    text = re.sub(r"<.*?>", "",text) 
    return text

df["review"] = df["review"].apply(remove_html)   

### 5. Removing the Stopwords

In [12]:
import nltk #natural language tool kit

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [16]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)       

In [17]:
df.head()

,review,sentiment
0,e revewers nte wtchg 1 oz epoe hooke ...,positive
1,wderful ltle prducti br br filming technique...,positive
2,hugh h werful w pen me h ummer weeken n...,positive
3,bcy fly e boy jke hk zobe cloe pn f...,negative
4,peer me love i vu unng film wch mr mei ...,positive


### 6. Steamming

In [18]:
# running -> run
# played -> play

from nltk.stem import PorterStemmer

In [19]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
        
    return " ".join(stemmed_words)    

df["review"] = df["review"].apply(stemming)           

### 7. Encoding

In [20]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [21]:
y = df["sentiment"]

In [22]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8.Vectorization

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

## Dataset and Data Loader

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2, random_state=42
) 

In [25]:
X_train.shape

(39665, 5000)

In [26]:
X_test.shape

(9917, 5000)

In [27]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [28]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [29]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [30]:
train_loader = DataLoader(train_set, shuffle=True,batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build our RNN

In [31]:
import torch.nn as nn
import torch.optim as optim

In [32]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first = True)

        #Fully connected layer
        self.fc = nn.Linear(hidden_size,1)

    def forward(self, x):
        #optional => shape(no of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0),self.hidden_size)

        out, _=self.rnn(x,h0)
        #1st value = hidden state of all the timestamps =>(batch size , seq_len ,hidden_size)
        #2nd value = final hidden state of last timestamps

        out = self.fc(out[:,-1,:])
        return out

In [33]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [34]:
epochs = 10
for epoch in range(epochs):
    model.train()

    for Xb,yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) #add singleton direction

        outputs = model(Xb) #(batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) #(batch_size,)=> probability

        loss = criterion(outputs, yb)
        loss.backward() #backprop
        optimizer.step() #weights update

    print(f"epoch ={epoch*1}/{epochs} and loss = {loss.item()}")    

epoch =0/10 and loss = 0.4012000560760498
epoch =1/10 and loss = 0.3475644588470459
epoch =2/10 and loss = 0.2467978298664093
epoch =3/10 and loss = 0.34161049127578735
epoch =4/10 and loss = 0.4092174768447876
epoch =5/10 and loss = 0.2944660186767578
epoch =6/10 and loss = 0.288297176361084
epoch =7/10 and loss = 0.4101046323776245
epoch =8/10 and loss = 0.264245867729187
epoch =9/10 and loss = 0.2616640329360962


### evaluation

In [37]:
# evaluation

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals / tot_vals * 100}")

accuracy = 82.69637995361501
